# 1. Download data from Don'tGetKicked competiton.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_selector, ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from category_encoders import CountEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, make_scorer
from sklearn.pipeline import Pipeline
from tqdm import tqdm
from sklearn.model_selection import GridSearchCV

In [2]:
df = pd.read_csv('./data/training.csv', parse_dates=['PurchDate'])

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   RefId                              72983 non-null  int64         
 1   IsBadBuy                           72983 non-null  int64         
 2   PurchDate                          72983 non-null  datetime64[ns]
 3   Auction                            72983 non-null  object        
 4   VehYear                            72983 non-null  int64         
 5   VehicleAge                         72983 non-null  int64         
 6   Make                               72983 non-null  object        
 7   Model                              72983 non-null  object        
 8   Trim                               70623 non-null  object        
 9   SubModel                           72975 non-null  object        
 10  Color                             

# 2. Design the train/validation/test split.

In [4]:
df = df.sort_values('PurchDate')
df.head()

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
32367,32389,0,2009-01-05,MANHEIM,2007,2,CHRYSLER,PACIFICA FWD 3.8L V6,Bas,4D SPORT,...,9906.0,11657.0,NaN,NaN,3453,80022,CO,6770.0,0,1389
32384,32406,0,2009-01-05,MANHEIM,2005,4,FORD,FREESTAR FWD V6 3.9L,SES,4D PASSENGER 3.9L SES,...,5801.0,6949.0,NaN,NaN,22916,80022,CO,6160.0,0,941
32385,32407,0,2009-01-05,MANHEIM,2004,5,DODGE,STRATUS 4C 2.4L I4 M,SE,4D SEDAN SE,...,4169.0,5114.0,NaN,NaN,3453,80022,CO,4250.0,0,1155
32386,32408,0,2009-01-05,MANHEIM,2006,3,CHEVROLET,TRAILBLAZER EXT 4WD,LS,4D SUV 4.2L,...,10438.0,12158.0,NaN,NaN,22916,80022,CO,8180.0,0,1703
32387,32409,0,2009-01-05,MANHEIM,2004,5,FORD,TAURUS 3.0L V6 EFI,SES,4D SEDAN SES DURATEC,...,4139.0,5351.0,NaN,NaN,22916,80022,CO,4900.0,0,825


In [5]:
train_valid_df, test_df = train_test_split(df, test_size=0.33333, shuffle=False)
train_df, valid_df = train_test_split(train_valid_df, test_size=0.5, shuffle=False)

In [6]:
print(len(train_df))
print(len(valid_df))
print(len(test_df))

24327
24328
24328


In [7]:
X_train = train_df.drop(columns=['IsBadBuy', 'PurchDate'])
y_train = train_df['IsBadBuy']

X_valid = valid_df.drop(columns=['IsBadBuy', 'PurchDate'])
y_valid = valid_df['IsBadBuy']

X_test = test_df.drop(columns=['IsBadBuy', 'PurchDate'])
y_test = test_df['IsBadBuy']

# 3. Use LabelEncoder or OneHotEncoder from sklearn to preprocess categorical variables.

In [8]:
train_df.columns

Index(['RefId', 'IsBadBuy', 'PurchDate', 'Auction', 'VehYear', 'VehicleAge',
       'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission',
       'WheelTypeID', 'WheelType', 'VehOdo', 'Nationality', 'Size',
       'TopThreeAmericanName', 'MMRAcquisitionAuctionAveragePrice',
       'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice',
       'MMRAcquisitonRetailCleanPrice', 'MMRCurrentAuctionAveragePrice',
       'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice',
       'MMRCurrentRetailCleanPrice', 'PRIMEUNIT', 'AUCGUART', 'BYRNO',
       'VNZIP1', 'VNST', 'VehBCost', 'IsOnlineSale', 'WarrantyCost'],
      dtype='object')

In [9]:
train_df.IsBadBuy.unique()

array([0, 1])

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 72983 entries, 32367 to 70421
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   RefId                              72983 non-null  int64         
 1   IsBadBuy                           72983 non-null  int64         
 2   PurchDate                          72983 non-null  datetime64[ns]
 3   Auction                            72983 non-null  object        
 4   VehYear                            72983 non-null  int64         
 5   VehicleAge                         72983 non-null  int64         
 6   Make                               72983 non-null  object        
 7   Model                              72983 non-null  object        
 8   Trim                               70623 non-null  object        
 9   SubModel                           72975 non-null  object        
 10  Color                              

In [11]:
cat_selector = make_column_selector(dtype_include=['object', 'category'])
cat_columns = cat_selector(train_df)

Список всех столбцов, в которых значения не числовые, а объектные.

In [12]:
cat_columns

['Auction',
 'Make',
 'Model',
 'Trim',
 'SubModel',
 'Color',
 'Transmission',
 'WheelType',
 'Nationality',
 'Size',
 'TopThreeAmericanName',
 'PRIMEUNIT',
 'AUCGUART',
 'VNST']

In [13]:
train_df.VNST.unique()

array(['CO', 'NC', 'CA', 'TX', 'FL', 'UT', 'IN', 'AZ', 'OK', 'SC', 'GA',
       'LA', 'IA', 'VA', 'TN', 'MD', 'MS', 'AL', 'MO', 'OH', 'IL', 'ID',
       'NM', 'PA', 'NV', 'NE', 'NJ', 'KY', 'WA'], dtype=object)

### Анализ каждого признака:
* Auction - поставщик аукциона, на котором было приобретено транспортное стредство. Данные признак будет являться категориальным, так как явного порядко тут задать не получится.
* Make - марка машины. Тоже будет категориальным, хотя и можно было бы попытаться придумать порядок, но будет излишним.
* Model - модель машины. Категорильный признак.
* Trim - уровень отделки салона. Скорее всего уровень указан для разных моделей и марок разный, поэтому порядок установть будет сложно, и может не так критично, как если просто оставить категорилаьными. Хотя если очень глубоко изучить, то можно и составить порядок.
* SubModel - подмодель автомобила. Категорильный признак.
* Color - цвет. Категорильный признак.
* Transmision - коробка передач, то есть автомат или ручная. Можно сказать, что это порядковый, но так как тут всего 2 варианта, то и обычный OneHotEncoder даст то же самое.
* WheeType - тип колёс. Может и можно было бы придумать порядок, но очень сомнительно. Обычный категорильаный признак.
* Natinality - национальнось, или же регион или страна производителя. Категорильный признак.
* Size - скорее имеется ввиду тип кузова, но как бы размер. С одной стороны можно было порядок ввести, однако например тип SPORTS не получится объединить с другими типами, так как это скорее про тип кузова, так что тут тоже категорильный.
* TopThreeAmericanName - определяет, входит ли производитель в топ 3 американских производителей. Категорильный признак.
* PRIMEUNIT - определяет, будет ли спрос на транспортное средство выше, чем при стандартной покупке. Категорильный признак. Хотя тут значения либо YES либо пропещенный, поэтому тут порядка и не получитя придумать. А ещё супер много пропущенных значений, может быть лучше вообще выбросить этот признак. Всех nan на 0 не факт, что поможет, хотя в целом можно и будте оставить, просто где есть ответ, там потавить 1.
* AUCGUART - уровень гарантии, предоставляемой на аукционе. GREEN - значит всё хорошо, остальных вариантов в train нет, и может быть в других частях есть, но не факт, также много пропущенных значений, хотя вот тут как раз и можно было бы применить порядковое.
* VNST - штат, в котором приобрели автомобиль. Категорильный признак.

Теперь надо разобраться с тем, что делать с пропущенными значениями. Стоит ли придумать на что их заменить или убрать те признаки, в которых пропущенно очень много значений.

In [14]:
X_train.isna().sum()[X_train.isna().sum() > 0]

Trim                                   727
SubModel                                 8
Color                                    8
Transmission                             8
WheelTypeID                            925
WheelType                              925
MMRAcquisitionAuctionAveragePrice        3
MMRAcquisitionAuctionCleanPrice          3
MMRAcquisitionRetailAveragePrice         3
MMRAcquisitonRetailCleanPrice            3
MMRCurrentAuctionAveragePrice          260
MMRCurrentAuctionCleanPrice            260
MMRCurrentRetailAveragePrice           260
MMRCurrentRetailCleanPrice             260
PRIMEUNIT                            24325
AUCGUART                             24325
dtype: int64

Сразу стоит сказать , что для признаков PRIMEUNIT и AUCGART пропуссков больше 90% от всей тренировочнйо выборки, при этом предположение о том, что если значение nan, то оно обратное, так как это поля про то, что  будет ли спрос выше и про гарантию, то предполгать обратное может быть ошибочным. Поэтому лучше будет избавиться от этих признак вовсе.

In [15]:
X_train['Trim'].value_counts().iloc[0]

np.int64(4540)

Для поля Trim думаю будет оптимальным вариантом для пропущенных значения установить значеиние Bas, то есть базовые обычные.

In [16]:
X_train['SubModel'].value_counts()

SubModel
4D SEDAN              4903
4D SEDAN SE           1427
4D SEDAN LS           1259
4D SEDAN SXT FFV       742
MINIVAN 3.3L           517
                      ... 
PASSENGER 3.4L LS        1
MEGA CAB 5.7L            1
REG CAB 6.0L LS          1
EXT CAB 4.7L             1
PASSENGER EXT 3.9L       1
Name: count, Length: 663, dtype: int64

Для SUbModel, Color, Transmission просто заменю Nan на самые часто встречающиеся значения.

In [17]:
X_train['WheelTypeID'].value_counts()

WheelTypeID
1.0    12545
2.0    10622
3.0      235
Name: count, dtype: int64

Для колес оба поля Nan заменю на самые частво встречющиеся.

Остальные числовые признаки заменю срденими значениями.

In [18]:
X_train.isna().sum()[X_train.isna().sum() > 0].index

Index(['Trim', 'SubModel', 'Color', 'Transmission', 'WheelTypeID', 'WheelType',
       'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice',
       'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice',
       'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice',
       'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice',
       'PRIMEUNIT', 'AUCGUART'],
      dtype='object')

In [19]:
X_train[X_train['Trim'].isna()]

,RefId,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
8866,8875,MANHEIM,2004,5,KIA,AMANTI 3.5L V6 MPI,NaN,4D SEDAN,BLUE,AUTO,...,5369.0,6421.0,NaN,NaN,17675,27542,NC,5535.0,0,1118
32377,32399,MANHEIM,2004,5,CHRYSLER,PACIFICA AWD 3.5L V6,NaN,4D SPORT TOURER,WHITE,AUTO,...,8311.0,9676.0,NaN,NaN,22916,80022,CO,6010.0,0,2022
33243,33265,MANHEIM,2005,4,SATURN,RELAY 2WD V6 3.5L V6,NaN,4D SPORT UTILITY,BLUE,AUTO,...,6736.0,8247.0,NaN,NaN,5546,34761,FL,5355.0,0,2282
33242,33264,MANHEIM,2002,7,LINCOLN,CONTINENTAL 4.6L V8,NaN,4D SEDAN,SILVER,AUTO,...,4763.0,5743.0,NaN,NaN,21973,34761,FL,5000.0,0,1572
33238,33260,MANHEIM,2007,2,SUZUKI,FORENZA 2.0L I4 EFI,NaN,4D SEDAN,RED,MANUAL,...,6620.0,7376.0,NaN,NaN,19638,34761,FL,4750.0,0,482
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19009,19022,MANHEIM,2004,5,CHRYSLER,PACIFICA AWD 3.5L V6,NaN,4D SPORT TOURER,SILVER,AUTO,...,8087.0,9831.0,NaN,NaN,22916,80022,CO,8380.0,0,2206
19013,19026,MANHEIM,2002,7,LEXUS,ES300 3.0L V6 EFI,NaN,4D SEDAN,SILVER,AUTO,...,9976.0,11511.0,NaN,NaN,22916,80022,CO,10190.0,0,1497
59536,59565,ADESA,2007,2,SUZUKI,FORENZA 2.0L I4 EFI,NaN,4D SEDAN,RED,AUTO,...,6741.0,7695.0,NaN,NaN,18822,78754,TX,5800.0,0,482
13522,13532,MANHEIM,2001,8,LEXUS,GS300 3.0L I6 EFI,NaN,4D SEDAN,SILVER,AUTO,...,9324.0,11159.0,NaN,NaN,16044,21075,MD,8185.0,0,1413


In [20]:
def remove_nan_values(X_train, X_valid, X_test):
     # убираем 2 столбца с пропущенными значениями
    X_train.drop(columns=['PRIMEUNIT', 'AUCGUART'], inplace=True)
    X_valid.drop(columns=['PRIMEUNIT', 'AUCGUART'], inplace=True)
    X_test.drop(columns=['PRIMEUNIT', 'AUCGUART'], inplace=True)

    features_top = ['Trim', 'SubModel', 'Color', 'Transmission', 'WheelTypeID', 'WheelType']
    features_mean = ['MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice',
       'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice',
       'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice',
       'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice']
    
    for feature in features_top:
        base = X_train[feature].value_counts().iloc[0]

        X_train.loc[X_train[feature].isna(), feature] = base
        X_valid.loc[X_valid[feature].isna(), feature] = base
        X_test.loc[X_test[feature].isna(), feature] = base

    for feature in features_mean:
        base = X_train[feature].mean()

        X_train.loc[X_train[feature].isna(), feature] = base
        X_valid.loc[X_valid[feature].isna(), feature] = base
        X_test.loc[X_test[feature].isna(), feature] = base
    
    return X_train, X_valid, X_test

In [21]:
X_train, X_valid, X_test = remove_nan_values(X_train, X_valid, X_test)

In [22]:
y_valid.drop(index=X_valid[X_valid['Size'].isna()].index, inplace=True)
X_valid.drop(index=X_valid[X_valid['Size'].isna()].index, inplace=True)
X_valid.isna().sum().sum()

np.int64(0)

С пропусками разобрались, теперь применим encoder

In [23]:
categorical_columns = ['Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission', 'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName', 'VNST']

In [24]:
for col in categorical_columns:
    X_train[col] = X_train[col].astype(str)
    X_valid[col] = X_valid[col].astype(str)
    X_test[col] = X_test[col].astype(str)

In [25]:
numeric_columns = [column for column in X_train.columns if column not in categorical_columns]
numeric_columns

['RefId',
 'VehYear',
 'VehicleAge',
 'WheelTypeID',
 'VehOdo',
 'MMRAcquisitionAuctionAveragePrice',
 'MMRAcquisitionAuctionCleanPrice',
 'MMRAcquisitionRetailAveragePrice',
 'MMRAcquisitonRetailCleanPrice',
 'MMRCurrentAuctionAveragePrice',
 'MMRCurrentAuctionCleanPrice',
 'MMRCurrentRetailAveragePrice',
 'MMRCurrentRetailCleanPrice',
 'BYRNO',
 'VNZIP1',
 'VehBCost',
 'IsOnlineSale',
 'WarrantyCost']

In [26]:
for col in categorical_columns:
    print(f'\nFor {col}: \n')
    print(X_valid[col].unique())


For Auction: 

['MANHEIM' 'OTHER' 'ADESA']

For Make: 

['CHRYSLER' 'FORD' 'CHEVROLET' 'SATURN' 'BUICK' 'DODGE' 'MAZDA' 'HYUNDAI'
 'KIA' 'SCION' 'MERCURY' 'PONTIAC' 'JEEP' 'OLDSMOBILE' 'TOYOTA' 'NISSAN'
 'SUZUKI' 'GMC' 'HONDA' 'LINCOLN' 'MITSUBISHI' 'INFINITI' 'LEXUS' 'ACURA'
 'ISUZU' 'VOLKSWAGEN' 'MINI' 'TOYOTA SCION' 'SUBARU' 'CADILLAC' 'VOLVO']

For Model: 

['PT CRUISER 2.4L I4 S' 'ESCAPE 4WD V6 3.0L V' 'TOWN & COUNTRY 2WD V'
 '1500 SILVERADO PICKU' 'EXPEDITION 2WD V8 4.' 'VUE 2WD 4C 2.2L I4 M'
 'COBALT 2.2L I4 MPI' 'MALIBU V6 3.5L V6 SF' 'RENDEZVOUS AWD 3.4L'
 '1500 RAM PICKUP 2WD' 'MALIBU 4C 2.2L I4 MP' 'PACIFICA FWD 3.5L V6'
 'NEON 2.0L I4 SFI' 'UPLANDER FWD V6 3.9L' 'STRATUS 4C 2.4L I4 S'
 'DURANGO 2WD V6 3.7L' 'CARAVAN FWD V6 3.3L' 'MAZDA6 2.3L I4 MFI /'
 'MAGNUM V6 2.7L V6 MP' 'F150 PICKUP 2WD V6 4' 'FOCUS 2.3L I4 DOHC'
 'IMPALA 3.5L V6 SFI' 'ELANTRA 2.0L I4 MPI' 'ION 2.0L I4 MPI'
 'AVENGER 4C 2.4L I4 S' 'CAVALIER 4C 2.2L I4' 'SORENTO 4WD 3.5L V6'
 'TIBURON MFI I-4 2.0L' 'TC

In [27]:
X_valid.loc[X_valid.Transmission == 'Manual', 'Transmission'] = 'MANUAL'

Make и ещё признаки убрали из категорильаных, так как для него надо другой encoder применить, ибо в valid есть значения , которых в train нет.

In [28]:
categorical_columns.remove('Make')
categorical_columns.remove('Model')
categorical_columns.remove('Trim')
categorical_columns.remove('SubModel')
categorical_columns.remove('VNST')

transformers = [
    ('num', StandardScaler(), numeric_columns),
    ('cat', OneHotEncoder(), categorical_columns),
    ('count_scaled', Pipeline([
        ('count', CountEncoder()),
        ('scaler', StandardScaler())
    ]), ['Make', 'Model', 'Trim', 'SubModel', 'VNST'])
]

ct = ColumnTransformer(transformers, remainder='drop')
X_train_transform = ct.fit_transform(X_train)
X_valid_transform = ct.transform(X_valid)
X_test_transform = ct.transform(X_test)

In [29]:
pd.DataFrame(X_train_transform, columns=ct.get_feature_names_out())

,num__RefId,num__VehYear,num__VehicleAge,num__WheelTypeID,num__VehOdo,num__MMRAcquisitionAuctionAveragePrice,num__MMRAcquisitionAuctionCleanPrice,num__MMRAcquisitionRetailAveragePrice,num__MMRAcquisitonRetailCleanPrice,num__MMRCurrentAuctionAveragePrice,...,cat__Size_VAN,cat__TopThreeAmericanName_CHRYSLER,cat__TopThreeAmericanName_FORD,cat__TopThreeAmericanName_GM,cat__TopThreeAmericanName_OTHER,count_scaled__Make,count_scaled__Model,count_scaled__Trim,count_scaled__SubModel,count_scaled__VNST
0,-0.241741,1.357265,-1.357240,-0.198594,0.608776,0.586988,0.656821,0.585911,0.655256,1.152128,...,0.0,1.0,0.0,0.0,0.0,-0.370581,-0.646739,1.495516,-0.519523,-0.303717
1,-0.240930,0.109884,-0.109905,-0.199010,-2.251226,-0.556616,-0.464911,-0.550313,-0.459592,-0.375865,...,1.0,0.0,1.0,0.0,0.0,0.383908,0.094845,-1.040652,-0.646551,-0.303717
2,-0.240882,-0.513807,0.513762,-0.198594,0.128598,-1.082305,-1.065826,-1.072680,-1.056844,-0.983284,...,0.0,1.0,0.0,0.0,0.0,0.647220,-0.113505,0.838189,0.090533,-0.303717
3,-0.240835,0.733574,-0.733573,-0.199010,-0.027052,1.095836,1.025344,1.091309,1.021473,1.350313,...,0.0,0.0,0.0,1.0,0.0,1.090292,-0.802119,0.574783,-0.576632,-0.303717
4,-0.240787,-0.513807,0.513762,-0.199010,-0.231833,-1.083909,-1.005124,-1.074156,-0.996689,-0.994539,...,0.0,0.0,1.0,0.0,0.0,0.383908,2.376099,-1.040652,-0.640146,-0.303717
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24322,0.926504,0.109884,-0.109905,-0.198594,0.048183,-1.061454,-1.083785,-1.052022,-1.074692,-1.083381,...,0.0,0.0,0.0,1.0,0.0,1.090292,-0.756211,0.574783,1.945785,0.082387
24323,0.926456,1.980956,-1.980908,-0.198594,-0.387833,1.126711,0.833898,1.122297,0.831093,0.853042,...,0.0,1.0,0.0,0.0,0.0,0.647220,-0.021690,0.838189,1.945785,0.082387
24324,0.926409,1.980956,-1.980908,-0.198594,-0.372226,1.126711,0.833898,1.122297,0.831093,1.095447,...,0.0,1.0,0.0,0.0,0.0,0.647220,-0.021690,0.838189,1.945785,0.082387
24325,-1.030312,0.733574,-0.733573,-0.198594,0.586100,1.164805,1.253066,1.159925,1.247550,0.702293,...,0.0,0.0,0.0,0.0,1.0,-1.674985,-0.915122,-1.156930,-0.638545,1.523458


# 4. Train LogisticRegression, GaussianNB, KNN from sklewrn on the traning dataset and chek the quality of your algorithms on the validation  dataset.

In [30]:
log_reg = LogisticRegression()
log_reg.fit(X_train_transform, y_train)

knn = KNeighborsClassifier()
knn.fit(X_train_transform, y_train)

naive_baues = GaussianNB()
naive_baues.fit(X_train_transform, y_train)

,priors,None
,var_smoothing,1e-09


In [31]:
def my_simple_gini(y_true, y_preduct):
    return abs(2*roc_auc_score(y_true, y_preduct) - 1)

In [32]:
log_reg_gini = my_simple_gini(y_valid, log_reg.predict_proba(X_valid_transform)[:,1])
knn_gini = my_simple_gini(y_valid, knn.predict_proba(X_valid_transform)[:,1])
naive_baues_gini = my_simple_gini(y_valid, naive_baues.predict_proba(X_valid_transform)[:,1])

print(f'Logistic Regression gini: {log_reg_gini}')
print(f'KNN gini: {knn_gini}')
print(f'Gaussian Naive Bauys gini: {naive_baues_gini}')

Logistic Regression gini: 0.4226569656305825
KNN gini: 0.3411520019186498
Gaussian Naive Bauys gini: 0.41939355200516903


Лучше всего получилось у логистической регрессии. Скорее всего для KNN надо лучше подбирать гиперпараметры, а на классических получается очень плохой результат. Гауссовский наивный байес справился чуть хуже, может быть из-за зависимости признаков или просто распредления признаков не совсем нормальными являются.

# 5. Implement Gini score colculation. You can use the 2*ROC AUC - 1 approach, so you need to implement the ROC AYC calculation.

In [33]:
def calculate_roc_auc(y_true, y_score):
    data = list(zip(y_score, y_true))
    data_sorted = sorted(data, key=lambda x: x[0], reverse=True)

    thresholds = sorted(set(y_score), reverse=True)
    thresholds.append(1.0)
    thresholds = sorted(thresholds, reverse=True)

    P = sum(y_true) # всего реальных 1
    N = len(y_true) - P # всего реальных 0

    fpr_points = []
    tpr_points = []

    for threshold in thresholds:
        TP = 0
        FP = 0

        for score, true_class in data_sorted:
            if score >= threshold:
                if true_class == 1:
                    TP += 1
                else:
                    FP += 1
        
        TPR = TP / P if P > 0 else 0
        FPR = FP / N if N > 0 else 0

        fpr_points.append(FPR)
        tpr_points.append(TPR)
    
    fpr_points.append(1.0)
    tpr_points.append(1.0)

    points = sorted(zip(fpr_points, tpr_points), key=lambda x: x[0])
    fpr_sorted, tpr_sorted = zip(*points)

    auc = 0.0
    for i in range(1, len(fpr_points)):
        width = fpr_sorted[i] - fpr_sorted[i-1]
        avg_height = (tpr_sorted[i] + tpr_sorted[i-1]) / 2
        auc += width * avg_height

    return auc

In [34]:
y_score = log_reg.predict_proba(X_valid_transform)[:,1]
print(f'sklearn:  {roc_auc_score(y_valid, y_score)}')
print(f'my function: {calculate_roc_auc(y_valid.values, y_score)}')

sklearn:  0.7113284828152913
my function: 0.7113284828152948


In [35]:
def my_gini_score(y_true, y_preduct):
    return abs(2*calculate_roc_auc(y_true, y_preduct) - 1)

In [36]:
log_reg_gini = my_gini_score(y_valid, log_reg.predict_proba(X_valid_transform)[:,1])
knn_gini = my_gini_score(y_valid, knn.predict_proba(X_valid_transform)[:,1])
naive_baues_gini = my_gini_score(y_valid, naive_baues.predict_proba(X_valid_transform)[:,1])

print(f'Logistic Regression gini: {log_reg_gini}')
print(f'KNN gini: {knn_gini}')
print(f'Gaussian Naive Bauys gini: {naive_baues_gini}')

Logistic Regression gini: 0.42265696563058963
KNN gini: 0.3408471043604653
Gaussian Naive Bauys gini: 0.41294445779826905


значения получились близкие к тем, в которых roc_auc считался при помощи sklearn

# 6. Implement your own version of LogisticRegression, KNN and NaiveBayes classifier.

### LogisticRegression

In [37]:
class myLogisticRegression:
    def __init__(self, learning_rate=0.01, n_epochs=1000, batch_size=32, random_state=21, alpha=0.01):
        self.lr = learning_rate
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.w = None
        self.random_state = random_state
        self.alpha = alpha

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def fit(self, X_input, y):
        X = np.hstack((np.ones((X_input.shape[0], 1)), X_input)) # X с столбцом из единиц для свободного члена
        self.w = np.zeros(X.shape[1]) # вектор весов
        np.random.seed(self.random_state)

        for epoch in range(self.n_epochs):
            indices = np.arange(len(X))
            np.random.shuffle(indices)
            
            for i in range(0, len(X), self.batch_size):
                batch_indices = indices[i:i+self.batch_size]

                X_batch = X[batch_indices] # берём батч из всего X
                Y_batch = y[batch_indices]

                P_batch = self._sigmoid(X_batch @ self.w) # вычилсяем вероятности

                gradient = (1 / self.batch_size) * (X_batch.T @ (P_batch - Y_batch))

                self.w -= self.lr * gradient
        
    
    def predict_proba(self, X_input):
        X = np.hstack((np.ones((X_input.shape[0], 1)), X_input))
        P = self._sigmoid(X @ self.w)

        return P
    
    def predict(self, X_input, threshold=0.5):
        X = np.hstack((np.ones((X_input.shape[0], 1)), X_input))
        P = self._sigmoid(X @ self.w)


        predictions = (P >= threshold).astype(int)
        
        return predictions      


In [38]:
my_log_reg = myLogisticRegression()
my_log_reg.fit(X_train_transform, y_train.values)

In [39]:
my_log_reg_gini = my_simple_gini(y_valid, my_log_reg.predict_proba(X_valid_transform))

print(f'Logistic Regression gini: {log_reg_gini}')
print(f'My Logistic Regression gini: {my_log_reg_gini}')

Logistic Regression gini: 0.42265696563058963
My Logistic Regression gini: 0.43980173234152886


На валидационной выборке моя модель получила даже чуть выше значения, скорее всего из-за чуть лучшего подбора гиперпараметров и количсетва итераций.

### KNN

In [40]:
class my_KNN:
    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors

    def fit(self, X, y):
        self.train_features = X
        self.train_target = y

    def get_neighbors(self, X_point):
        differences = self.train_features - X_point
        distances = np.sqrt(np.sum(differences ** 2, axis=1))

        all_neighbors = np.column_stack([distances, self.train_target])

        sorted_indices = np.argsort(all_neighbors[:,0])[:self.n_neighbors]
        nearest_n_neighbors = all_neighbors[sorted_indices]

        return nearest_n_neighbors[:,1]

    def predict_proba(self, X):
        return [self.get_neighbors(point).mean() for point in tqdm(X, desc='KNN Prediction', total=len(X))]
    
    def predict(self, X):
        predictions = (self.predict_proba(X) >= 0.5).astype(int)
        
        return predictions

In [41]:
my_knn = my_KNN()
my_knn.fit(X_train_transform, y_train)

In [42]:
my_knn_predict_proba = my_knn.predict_proba(X_valid_transform)

KNN Prediction: 100%|██████████| 24323/24323 [05:03<00:00, 80.04it/s]


In [43]:
my_knn_gini = my_simple_gini(y_valid, my_knn_predict_proba)

print(f'KNN gini: {knn_gini}')
print(f'My KNN gini: {my_knn_gini}')

KNN gini: 0.3408471043604653
My KNN gini: 0.3411520019186498


### Guassian Naive Bayes

In [44]:
class my_GuassionNaiveBayes:
    def __init__(self, var_smoothing=1e-09):
        self.var_smoothing = var_smoothing

    def fit(self, X, y):
        n_0 = (y == 0).sum() # число объектов y = 0
        n_1 = (y == 1).sum() # число объектов y = 1

        self.P_y_0 = n_0 / (n_0 + n_1) # P(y=0)
        self.P_y_1 = n_1 / (n_0 + n_1) # априорные вероятности P(y=1)

        X_y_0 = X[y == 0]
        self.mean_y_0 = X_y_0.mean(axis=0)
        self.var_y_0 = X_y_0.var(axis=0) + self.var_smoothing

        X_y_1 = X[y == 1]
        self.mean_y_1 = X_y_1.mean(axis=0)
        self.var_y_1 = X_y_1.var(axis=0) + self.var_smoothing

    def predict_proba(self, X):
        score_0 = np.log(self.P_y_0) + np.sum((((-1/2) * np.log(2 * np.pi * self.var_y_0)) - (((X - self.mean_y_0)**2) / (2*self.var_y_0))), axis=1)
        score_1 = np.log(self.P_y_1) + np.sum((((-1/2) * np.log(2 * np.pi * self.var_y_1)) - (((X - self.mean_y_1)**2) / (2*self.var_y_1))), axis=1)

        max_score = np.maximum(score_0, score_1)
        exp_score_0 = np.exp(score_0 - max_score)
        exp_score_1 = np.exp(score_1 - max_score)

        proba_1 = exp_score_1 / (exp_score_0 + exp_score_1)
        return proba_1
    
    def predict(self, X):
        predictions = (self.predict_proba(X) >= 0.5).astype(int)
        return predictions


In [45]:
my_guassianNB = my_GuassionNaiveBayes()
my_guassianNB.fit(X_train_transform, y_train)

In [46]:
my_guassianNB.predict_proba(X_valid_transform)

array([9.11075755e-03, 1.39697313e-03, 2.08713693e-04, ...,
       5.90294596e-05, 9.82326968e-05, 3.71502042e-02], shape=(24323,))

In [47]:
print(f'GuassianNB gini: {naive_baues_gini}')
print(f'My guassianNB gini: {my_simple_gini(y_valid, my_guassianNB.predict_proba(X_valid_transform))}')

GuassianNB gini: 0.41294445779826905
My guassianNB gini: 0.419421633888835


# 7. Try to create non-linear features, for example:
- fractions: feature1 / feature2
- groupby features: df[‘categorical_feature’].map(df.groupby(‘categorical_feature’)[‘continious_feature’].mean())

Add new features to your pipeline, repeat step 4. Did you manage to increase your Gini score (you should!)?

In [48]:
numeric_selector = make_column_selector(dtype_include=['int64', 'float64'])
numeric_columns_for_make_non_linear = numeric_selector(X_train)
numeric_columns_for_make_non_linear

['RefId',
 'VehYear',
 'VehicleAge',
 'WheelTypeID',
 'VehOdo',
 'MMRAcquisitionAuctionAveragePrice',
 'MMRAcquisitionAuctionCleanPrice',
 'MMRAcquisitionRetailAveragePrice',
 'MMRAcquisitonRetailCleanPrice',
 'MMRCurrentAuctionAveragePrice',
 'MMRCurrentAuctionCleanPrice',
 'MMRCurrentRetailAveragePrice',
 'MMRCurrentRetailCleanPrice',
 'BYRNO',
 'VNZIP1',
 'VehBCost',
 'IsOnlineSale',
 'WarrantyCost']

Функция для деления одного признака на другой, тут есть условие, что не должны быть inf или все 0 или все одинаковые значения

In [49]:
def division_of_fearures(X, numeric_columns):   
    X_non_linear = X.copy()
    new_columns = {}

    for i in range(len(numeric_columns)):
        for j in range(len(numeric_columns)):
            if i != j:
                col_name = f'{numeric_columns[i]}/{numeric_columns[j]}'
                new_column = X_non_linear[numeric_columns[i]] / X_non_linear[numeric_columns[j]]
                
                new_columns[col_name] = new_column
                
    new_df = pd.DataFrame(new_columns).fillna(0)
    X_non_linear = pd.concat([X_non_linear, new_df], axis=1)

    return X_non_linear

Функция для доабвления средних по катогрии или же по категорильному признаку.

In [50]:
cat_columns_for_make_non_linear = cat_selector(X_train)
cat_columns_for_make_non_linear

['Auction',
 'Make',
 'Model',
 'Trim',
 'SubModel',
 'Color',
 'Transmission',
 'WheelType',
 'Nationality',
 'Size',
 'TopThreeAmericanName',
 'VNST']

In [51]:
class goupby_features:
    def __init__(self):
        pass

    def fit(self, X, cat_columns, numeric_columns):
        self.cat_columns = cat_columns
        self.numeric_columns = numeric_columns
        self.cat_means_all = {}
        self.global_means = {}

        for numeric in self.numeric_columns:
            self.global_means[numeric] = X[numeric].mean()

        for category in self.cat_columns:
            for numeric in self.numeric_columns:
                key = f'{category}_{numeric}'
                category_means = X.groupby(category)[numeric].mean()
                self.cat_means_all[key] = category_means
    
    def transform(self, X):
        X_goruped_features = X.copy()
        new_columns = {}

        for category in self.cat_columns:
            for numeric in self.numeric_columns:
                key = f'{category}_{numeric}'
                new_col_name = f'{category}_{numeric}_mean'

                global_mean = self.global_means[numeric]
                new_columns[new_col_name] = X_goruped_features[category].map(self.cat_means_all[key]).fillna(global_mean)

        new_columns_df = pd.DataFrame(new_columns, index=X_goruped_features.index)
        X_goruped_features = pd.concat([X_goruped_features, new_columns_df], axis=1)

        return X_goruped_features        

### Добавление функция для создания нелинейных функций и применеие всех кодеров и нормализаций

Деление

In [52]:
X_train_fractions = division_of_fearures(X_train, numeric_columns_for_make_non_linear)
X_valid_fractions = division_of_fearures(X_valid, numeric_columns_for_make_non_linear)
X_test_fractions = division_of_fearures(X_test, numeric_columns_for_make_non_linear)

columns_to_del = []

for column in numeric_selector(X_train_fractions):
    if (X_train_fractions[column].nunique() <= 1) or ((X_train_fractions[column] == 0).all()) or (np.isinf(X_train_fractions[column]).any())\
        or (X_valid_fractions[column].nunique() <= 1) or ((X_valid_fractions[column] == 0).all()) or (np.isinf(X_valid_fractions[column]).any())\
        or (X_test_fractions[column].nunique() <= 1) or ((X_test_fractions[column] == 0).all()) or (np.isinf(X_test_fractions[column]).any()):
        columns_to_del.append(column)

X_train_fractions.drop(columns=columns_to_del, inplace=True)
X_valid_fractions.drop(columns=columns_to_del, inplace=True)
X_test_fractions.drop(columns=columns_to_del, inplace=True)

средние по категориям

In [53]:
numeric_columns_for_make_non_linear.remove('IsOnlineSale')
my_groupby_feature = goupby_features()
my_groupby_feature.fit(X_train_fractions, cat_columns_for_make_non_linear, numeric_columns_for_make_non_linear)

X_train_non_linear = my_groupby_feature.transform(X_train_fractions)
X_valid_non_linear = my_groupby_feature.transform(X_valid_fractions)
X_test_non_linear = my_groupby_feature.transform(X_test_fractions)

In [54]:
numeric_columns_after_addedd_nonlinear = numeric_selector(X_train_non_linear)

In [55]:
transformers = [
    ('num', StandardScaler(), numeric_columns_after_addedd_nonlinear),
    ('cat', OneHotEncoder(), categorical_columns),
    ('count_scaled', Pipeline([
        ('count', CountEncoder()),
        ('scaler', StandardScaler())
    ]), ['Make', 'Model', 'Trim', 'SubModel', 'VNST'])
]

ct = ColumnTransformer(transformers, remainder='drop')
X_train_transform_non_linear = ct.fit_transform(X_train_non_linear)
X_valid_transform_non_linear = ct.transform(X_valid_non_linear)
X_test_transform_non_linear = ct.transform(X_test_non_linear)

feature_names = ct.get_feature_names_out()

In [56]:
log_reg_non_linear = LogisticRegression()
log_reg_non_linear.fit(X_train_transform_non_linear, y_train)

knn_non_linear = KNeighborsClassifier()
knn_non_linear.fit(X_train_transform_non_linear, y_train)

naive_baues_non_linear = GaussianNB()
naive_baues_non_linear.fit(X_train_transform_non_linear, y_train)

c:\Users\User\anaconda3\envs\ml_projects\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,priors,None
,var_smoothing,1e-09


In [57]:
log_reg_gini = my_simple_gini(y_valid, log_reg.predict_proba(X_valid_transform)[:,1])
knn_gini = my_simple_gini(y_valid, knn.predict_proba(X_valid_transform)[:,1])
naive_baues_gini = my_simple_gini(y_valid, naive_baues.predict_proba(X_valid_transform)[:,1])

print(f'Результаты из пунка 4:')
print(f'Logistic Regression gini: {log_reg_gini}')
print(f'KNN gini: {knn_gini}')
print(f'Gaussian Naive Bauys gini: {naive_baues_gini}')

Результаты из пунка 4:
Logistic Regression gini: 0.4226569656305825
KNN gini: 0.3411520019186498
Gaussian Naive Bauys gini: 0.41939355200516903


In [58]:
log_reg_gini_non_linear = my_simple_gini(y_valid, log_reg_non_linear.predict_proba(X_valid_transform_non_linear)[:,1])
knn_gini_non_linear = my_simple_gini(y_valid, knn_non_linear.predict_proba(X_valid_transform_non_linear)[:,1])
naive_baues_gini_non_linear = my_simple_gini(y_valid, naive_baues_non_linear.predict_proba(X_valid_transform_non_linear)[:,1])

print(f'После добавления нелинейных признаков')
print(f'Logistic Regression gini: {log_reg_gini_non_linear}')
print(f'KNN gini: {knn_gini_non_linear}')
print(f'Gaussian Naive Bauys gini: {naive_baues_gini_non_linear}')

После добавления нелинейных признаков
Logistic Regression gini: 0.45923663641411316
KNN gini: 0.31519482643274843
Gaussian Naive Bauys gini: 0.3823841686528624


Для логистической регрессии улучшилось, для KNN и байеса наоборот узудшилось. Для наивного баеса так как признаки зависимые. Для knn просто из-за большоей размерности не всегда нелиненйные признаки делают лучше.

# 8. Determine the best features for the problem using the coefficiients of the logistic model. Try to aliminate unseless features by hand and by L1 regularization. Which approach is better it terms of Gini score ?

In [59]:
log_reg_l1 = LogisticRegression(penalty='l1', solver='liblinear')
log_reg_l1.fit(X_train_transform_non_linear, y_train)

,penalty,'l1'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [60]:
weights_feature = pd.DataFrame({
    'feature_name' : feature_names,
    'Importance' : np.abs(log_reg_l1.coef_[0])
})

In [61]:
weights_feature

,feature_name,Importance
0,num__RefId,0.092790
1,num__VehYear,0.433186
2,num__VehicleAge,0.000000
3,num__WheelTypeID,0.057553
4,num__VehOdo,0.000000
...,...,...
398,count_scaled__Make,0.070196
399,count_scaled__Model,0.151584
400,count_scaled__Trim,0.022279
401,count_scaled__SubModel,0.060890


In [62]:
weights_feature_sorted = weights_feature.sort_values('Importance', ascending=False)
weights_feature_sorted

,feature_name,Importance
51,num__MMRAcquisitionAuctionAveragePrice/VehOdo,1.160562
54,num__MMRAcquisitionAuctionAveragePrice/MMRAcqu...,1.083651
345,num__VNST_MMRCurrentRetailAveragePrice_mean,0.966717
79,num__MMRAcquisitonRetailCleanPrice/MMRAcquisit...,0.897190
81,num__MMRAcquisitonRetailCleanPrice/MMRAcquisit...,0.883221
...,...,...
211,num__Trim_BYRNO_mean,0.000000
216,num__SubModel_VehYear_mean,0.000000
220,num__SubModel_MMRAcquisitionAuctionAveragePric...,0.000000
41,num__WheelTypeID/VehBCost,0.000000


In [63]:
weights_feature_sorted[weights_feature_sorted['Importance'] > 0]

,feature_name,Importance
51,num__MMRAcquisitionAuctionAveragePrice/VehOdo,1.160562
54,num__MMRAcquisitionAuctionAveragePrice/MMRAcqu...,1.083651
345,num__VNST_MMRCurrentRetailAveragePrice_mean,0.966717
79,num__MMRAcquisitonRetailCleanPrice/MMRAcquisit...,0.897190
81,num__MMRAcquisitonRetailCleanPrice/MMRAcquisit...,0.883221
...,...,...
212,num__Trim_VNZIP1_mean,0.001137
167,num__Make_WheelTypeID_mean,0.000828
187,num__Model_MMRAcquisitionAuctionCleanPrice_mean,0.000724
255,num__Transmission_MMRAcquisitionAuctionCleanPr...,0.000655


In [64]:
top_100_features_indexes = weights_feature_sorted.index[:150]
top_100_features_indexes

Index([ 51,  54, 345,  79,  81, 348, 372,  78,  21, 203,
       ...
        28, 178, 229, 119, 231, 143, 201,  39,  40, 214],
      dtype='int64', length=150)

In [65]:
X_train_transform_non_linear[:,top_100_features_indexes].shape

(24327, 150)

In [66]:
X_train_important_features = X_train_transform_non_linear[:,top_100_features_indexes]
X_valid_important_features = X_valid_transform_non_linear[:,top_100_features_indexes]
X_test_important_features = X_test_transform_non_linear[:,top_100_features_indexes]

In [67]:
log_reg_important_features = LogisticRegression()
log_reg_important_features.fit(X_train_important_features, y_train)

knn_important_features = KNeighborsClassifier()
knn_important_features.fit(X_train_important_features, y_train)

naive_baues_important_features = GaussianNB()
naive_baues_important_features.fit(X_train_important_features, y_train)

c:\Users\User\anaconda3\envs\ml_projects\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,priors,None
,var_smoothing,1e-09


In [68]:
log_reg_gini_non_linear = my_simple_gini(y_valid, log_reg_non_linear.predict_proba(X_valid_transform_non_linear)[:,1])
knn_gini_non_linear = my_simple_gini(y_valid, knn_non_linear.predict_proba(X_valid_transform_non_linear)[:,1])
naive_baues_gini_non_linear = my_simple_gini(y_valid, naive_baues_non_linear.predict_proba(X_valid_transform_non_linear)[:,1])

print(f'По всем признакам')
print(f'Logistic Regression gini: {log_reg_gini_non_linear}')
print(f'KNN gini: {knn_gini_non_linear}')
print(f'Gaussian Naive Bauys gini: {naive_baues_gini_non_linear}')

По всем признакам
Logistic Regression gini: 0.45923663641411316
KNN gini: 0.31519482643274843
Gaussian Naive Bauys gini: 0.3823841686528624


In [69]:
log_reg_gini_important_features = my_simple_gini(y_valid, log_reg_important_features.predict_proba(X_valid_important_features)[:,1])
knn_gini_important_features = my_simple_gini(y_valid, knn_important_features.predict_proba(X_valid_important_features)[:,1])
naive_baues_gini_important_features = my_simple_gini(y_valid, naive_baues_important_features.predict_proba(X_valid_important_features)[:,1])

print(f'По топ 150 признакам')
print(f'Logistic Regression gini: {log_reg_gini_important_features}')
print(f'KNN gini: {knn_gini_important_features}')
print(f'Gaussian Naive Bauys gini: {naive_baues_gini_important_features}')

По топ 150 признакам
Logistic Regression gini: 0.453307385672826
KNN gini: 0.3142583553611811
Gaussian Naive Bauys gini: 0.3798022635073708


Самый высокий показатель gini  у логистичесокой регрессии.

# 9. Seelct your best model (algorithm + feature set) and tweak its hyperparameters to increase the Gini score on the validation dataset. Which hyperparameters have the modes impact ?

Модель с самым высоким показателем Gini - это Logistic Regression, и также я возьму топ 150 признаков по важности, полученные при помощи l1 регуляризации.

In [70]:
def gini_score(y_true, y_proba):
    auc = roc_auc_score(y_true, y_proba)
    return 2 * auc - 1

gini_scorer = make_scorer(
    gini_score,
    needs_proba=True,
    greater_is_better=True
)

In [71]:
param_grid = {
    'tol' : np.arange(0.0001, 0.001, 0.0001),
    'C' : [0.1, 1, 3, 10],
    'random_state' : [21],
    'solver' : ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
}

clf = GridSearchCV(LogisticRegression(max_iter=1000), param_grid=param_grid, n_jobs=-1, scoring='roc_auc')

In [72]:
clf.fit(X_train_important_features, y_train)

,estimator,LogisticRegre...max_iter=1000)
,param_grid,"{'C': [0.1, 1, ...], 'random_state': [21], 'solver': ['lbfgs', 'newton-cg', ...], 'tol': array([0.0001... 0.0009])}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [73]:
clf.best_params_

{'C': 0.1,
 'random_state': 21,
 'solver': 'saga',
 'tol': np.float64(0.0009000000000000001)}

In [74]:
best_model_log_reg = LogisticRegression(C= 0.1, random_state=21, solver='saga', tol=0.0009, max_iter=1000)

best_model_log_reg.fit(X_train_important_features, y_train)

best_gini = gini_score(y_valid, best_model_log_reg.predict_proba(X_valid_important_features)[:,1])

print(f'Gini for best model: {best_gini}')

Gini for best model: 0.4624243633881784


# 10. Check the Gini scores on all three datasets for your best model: traning Gini, valid Gini, test Gini. Do you see a drop in performance when comparing the valid quality to the test quality? Is your model overfittied or not ? Explain.

In [75]:
best_gini_train = gini_score(y_train, best_model_log_reg.predict_proba(X_train_important_features)[:,1])
best_gini_valid = gini_score(y_valid, best_model_log_reg.predict_proba(X_valid_important_features)[:,1])
best_gini_test = gini_score(y_test, best_model_log_reg.predict_proba(X_test_important_features)[:,1])


print(f'Train Gini: {best_gini_train}')
print(f'valid Gini: {best_gini_valid}')
print(f'test Gini: {best_gini_test}')

Train Gini: 0.5300220224315852
valid Gini: 0.4624243633881784
test Gini: 0.4786409339986981


Как видно по метрикам, модель не переобучена, так как на тестовой выборке метрика даже выше, чем на валидационной. Это говорит о том, что на данных, которые раньше модель не видела ни при отборе признаков, ни при подборе гиперпараметров, даже при преобработке данных все средние занчения или категории из теста ничего не бралось. И при этом метрика на тесте очень не плохая.

# 11. Implement calculation of Racall, Precision, F1 score and AUC PR metrics. Compare your algorithms on the test dataset using AUC PR metric.

In [76]:
def calculate_TP_TN_FP_FN(y_true, y_predict):
    TP = 0
    TN = 0
    FP = 0
    FN = 0

    for i in range(len(y_true)):
        if y_true[i] == 1:
            if y_predict == 1:
                TP += 1
            else:
                FN += 1
        elif y_true[i] == 0:
            if y_predict[i] == 0:
                TN += 1
            else:
                FP += 1

    return TP, TN, FP, FN

In [77]:
def my_precision(y_true, y_predict):
    TP, TN, FP, FN = calculate_TP_TN_FP_FN(y_true, y_predict)

    return TP / (TP + FP)

In [78]:
def my_recall(y_true, y_predict):
    TP, TN, FP, FN = calculate_TP_TN_FP_FN(y_true, y_predict)

    return TP / (TP + FN)

In [79]:
def my_F1_score(y_true, y_predict):
    precision = my_precision(y_true, y_predict)
    recall = my_recall(y_true, y_predict)

    return 2 * (precision * recall) / (precision + recall)

In [80]:
def calculate_auc_pr(y_true, y_proba):
    indices = np.argsort(y_proba)[::-1]
    y_true_sorted = y_true.iloc[indices]
    y_proba_sorted = y_proba[indices]

    precisions = []
    recalls = []
    tp = 0
    fp = 0
    fn = np.sum(y_true_sorted == 1)
    tn = np.sum(y_true_sorted == 0)

    for i in range(len(y_true_sorted)):
        if y_true_sorted.iloc[i] == 1:
            tp += 1
            fn -= 1
        else:
            fp += 1
            tn -= 1
        precision = tp / (tp + fp) if (tp + fp) > 1 else 1
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0

        precisions.append(precision)
        recalls.append(recall)

    auc_pr = 0
    for i in range(1, len(recalls)):
        auc_pr += (recalls[i] - recalls[i-1]) * (precisions[i] + precisions[i-1]) / 2

    return auc_pr

In [81]:
y_test.iloc[0]

np.int64(1)

In [82]:
log_reg_auc_pr_important_features = calculate_auc_pr(y_test, log_reg_important_features.predict_proba(X_test_important_features)[:,1])
knn_auc_pr_important_features = calculate_auc_pr(y_test, knn_important_features.predict_proba(X_test_important_features)[:,1])
naive_baues_auc_pr_important_features = calculate_auc_pr(y_test, naive_baues_important_features.predict_proba(X_test_important_features)[:,1])

print(f'По топ 150 важным признакам на test:')
print(f'Logistic Regression auc_pr: {log_reg_auc_pr_important_features}')
print(f'KNN auc_pr: {knn_auc_pr_important_features}')
print(f'Gaussian Naive Bauys auc_pr: {naive_baues_auc_pr_important_features}')

По топ 150 важным признакам на test:
Logistic Regression auc_pr: 0.4099215194994629
KNN auc_pr: 0.3688255754484181
Gaussian Naive Bauys auc_pr: 0.3189418115160606


# Which hard label metric do you prefer for the task of detecting "lemon" cars?

Я думаю неплохо будет использовать Recall, так как нам важнее найти все машины, которые истинно лимонног цвета, и если мы не лимонные тоже выберем, то не потеряем много, зато не упустим истинно лимонные, так как их не так уж и много.